# 🚀 LCR-SkateFormer: All-in-One Google Colab & Local Notebook
> **Mô hình**: **ECR-SkateFormer with KG-Guided Looping Concept Reasoning**
> **Mục tiêu nghiên cứu**: Khám phá và kiểm chứng giả thuyết rằng **semantic concepts trích xuất từ SkateFormer backbone có thể được tinh chỉnh qua nhiều vòng suy luận lặp (Iterative KG Reasoning)** trong concept/logit space ($z^{t+1} = z^t + \beta \Delta z^t$), tạo ra action evidence tiến bộ và nhất quán mà không làm loãng tính khả giải thích.

---

### 📌 Cấu trúc các khối trong Notebook (100% Self-Contained):
1. **Block 1**: Cài đặt Dependencies, Môi trường, Module Aliasing & Mount Google Drive
2. **Block 2**: Data Feeder, Augmentations Tools & Concept Ontology (72 Concepts)
3. **Block 3**: Kiến trúc SkateFormer Backbone
4. **Block 4**: Module Looping Reasoning (`KGUnionLayer`, `KGPrimedReasoner`, `ConceptUpdateCell`, `IterativeKGReasoner`, `LCRSkateFormer`, `LCR_SkateFormer`)
5. **Block 5**: Multi-Objective Looping Loss (Perception Loss $c^0$, Refined Loss $c^K$, Auxiliary Action Loss, Stability Loss, KG Consistency Loss)
6. **Block 6**: Trainer & Evaluator Engine (Checkpointing, Auto-Resume, Tensorboard)
7. **Block 7**: Pipeline Kiểm tra nhanh toàn diện với Dummy Data (Smoke Test & Trace Audit)
8. **Block 8**: Load Pretrained Checkpoint từ ECR-SkateFormer (Warm-up Phase 2)
9. **Block 9**: **Huấn luyện & Đánh giá với Dữ liệu Thật (NTU-RGB+D 60 / NTU-RGB+D 120)**
10. **Block 10**: Phân tích Giải thích & Can thiệp Phản thực (Counterfactual Intervention & Trace Audit)



## 📦 Block 1: Cài đặt Dependencies & Môi trường


In [1]:
# Cài đặt các thư viện cần thiết
import os
import sys
import math
import random
import time
import json
import csv
import glob
import types
import pickle
from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from tensorboardX import SummaryWriter
except ImportError:
    from torch.utils.tensorboard import SummaryWriter

class CosineLRScheduler:
    # Small timm-compatible cosine scheduler used by the notebook. The
    # notebook only needs step_update; keeping it local avoids importing timm,
    # which can trigger an incompatible torchvision build.
    def __init__(self, optimizer, t_initial, lr_min=0.0,
                 warmup_lr_init=0.0, warmup_t=0, t_in_epochs=False):
        self.optimizer = optimizer
        self.t_initial = max(1, int(t_initial))
        self.lr_min = float(lr_min)
        self.warmup_lr_init = float(warmup_lr_init)
        self.warmup_t = max(0, int(warmup_t))
        self.base_lrs = [float(group["lr"]) for group in optimizer.param_groups]

    def step_update(self, num_updates):
        step = max(0, int(num_updates))
        if self.warmup_t > 0 and step < self.warmup_t:
            alpha = step / float(self.warmup_t)
            lr_scale = None
        else:
            progress = (step - self.warmup_t) / float(
                max(1, self.t_initial - self.warmup_t)
            )
            progress = min(1.0, max(0.0, progress))
            lr_scale = 0.5 * (1.0 + math.cos(math.pi * progress))
            alpha = None

        for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            if lr_scale is None:
                lr = self.warmup_lr_init + alpha * (base_lr - self.warmup_lr_init)
            else:
                lr = self.lr_min + (base_lr - self.lr_min) * lr_scale
            group["lr"] = lr

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"⚡ Thiết bị sử dụng: {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")


⚡ Thiết bị sử dụng: cuda:0
🔥 GPU: NVIDIA GeForce RTX 5090


C:\Users\ADMIN\miniconda3\envs\reason\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📂 Block 2: Data Feeder, Augmentation Tools & Concept Ontology


In [2]:
# --- 1. Embedded Concept CSV (72 Concepts Ontology) ---
EMBEDDED_CONCEPTS_CSV = """action_class,head_nod,head_shake,head_tilt_down,head_tilt_up,head_rotate,head_static,hand_grasp,hand_release,hand_wave,hand_point,hand_clap,hand_manipulate,hand_fist,hand_open,hand_touch,hand_lift_to_face,hand_circular,arm_raise,arm_lower,arm_extend,arm_retract,arm_circular,arm_swing,arm_cross,arm_hold,arm_alternating,arm_overhead,arm_lateral,hip_bend,hip_extend,hip_sit,hip_squat,hip_rotate,hip_static_stand,hip_static_sit,hip_sway,leg_step,leg_kick,leg_jump,leg_squat,leg_lift,leg_static_stand,leg_static_sit,leg_alternating,leg_balance,foot_step,foot_kick,foot_jump,foot_pivot,foot_static,foot_tap,foot_slide,interaction_two_person,temporal_direction_motion_upward,temporal_direction_motion_downward,temporal_direction_motion_forward,temporal_direction_motion_backward,temporal_direction_motion_converging,temporal_direction_motion_diverging,temporal_direction_motion_reversible,temporal_sequence_seq_hands_first,temporal_sequence_seq_legs_first,temporal_sequence_seq_body_first,temporal_sequence_seq_simultaneous,temporal_sequence_seq_alternating,temporal_sequence_seq_cascading,temporal_dynamics_speed_slow,temporal_dynamics_speed_fast,temporal_dynamics_speed_accelerating,temporal_dynamics_speed_decelerating,temporal_dynamics_rhythm_regular,temporal_dynamics_rhythm_irregular,temporal_dynamics_duration_brief,temporal_dynamics_duration_sustained
drink_water,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0
eat_meal,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1
brush_teeth,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1
brush_hair,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1
drop,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0
pick_up,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0
throw,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
sit_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0
stand_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0
clapping,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1
reading,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
writing,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1
tear_up_paper,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0
put_on_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
take_off_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
put_on_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0
take_off_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0
put_on_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
take_off_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
put_on_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
take_off_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
cheer_up,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
hand_waving,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1
kicking_something,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
reach_into_pocket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
hopping,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
jump_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
phone_call,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
play_with_phone_tablet,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1
type_on_a_keyboard,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1
point_to_something,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
taking_a_selfie,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
check_time_(from_watch),0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
rub_two_hands,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1
nod_head_bow,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0
shake_head,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0
wipe_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0
salute,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
put_palms_together,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
cross_hands_in_front,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
sneeze_cough,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
staggering,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1
falling_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
headache,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
chest_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
back_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
neck_pain,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1
nausea_vomiting,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
fan_self,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1
punch_slap,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
kicking,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
pushing,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0
pat_on_back,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1
point_finger,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
hugging,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1
giving_object,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
touch_pocket,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
shaking_hands,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0
walking_towards,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
walking_apart,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
put_on_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
take_off_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
shoot_at_basket,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
bounce_ball,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1
tennis_bat_swing,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
juggle_table_tennis_ball,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1
hush,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
flick_hair,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
thumb_up,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
thumb_down,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
make_OK_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
make_victory_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
staple_book,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
counting_money,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
cutting_nails,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1
cutting_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
snap_fingers,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
open_bottle,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,1,0
sniff_smell,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0
squat_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0
toss_a_coin,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
fold_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
ball_up_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0
play_magic_cube,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1
apply_cream_on_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
apply_cream_on_hand,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
put_on_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
take_off_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0
put_object_into_bag,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
take_object_out_of_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
open_a_box,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0
move_heavy_objects,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
shake_fist,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0
throw_up_cap_hat,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
capitulate,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
cross_arms,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1
arm_circles,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1
arm_swings,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1
run_on_the_spot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1
butt_kicks,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1
cross_toe_touch,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1
side_kick,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
yawn,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1
stretch_oneself,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1
blow_nose,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
hit_with_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
wield_knife,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
knock_over,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
grab_stuff,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0
shoot_with_gun,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
step_on_foot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
high-five,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
cheers_and_drink,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0
carry_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
take_a_photo,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
follow,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
whisper,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
exchange_things,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
support_somebody,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
rock-paper-scissors,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,0
"""

os.makedirs("concepts", exist_ok=True)
CONCEPTS_CSV_PATH = "concepts/ntu_rgbd_60_120.csv"
if not os.path.exists(CONCEPTS_CSV_PATH):
    with open(CONCEPTS_CSV_PATH, "w", encoding="utf-8") as f:
        f.write(EMBEDDED_CONCEPTS_CSV.strip())
    print(f"✅ Đã tạo file concept ontology tại: {CONCEPTS_CSV_PATH}")
else:
    print(f"ℹ️ Sử dụng file concept ontology có sẵn tại: {CONCEPTS_CSV_PATH}")

def init_seed(seed=1):
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



ℹ️ Sử dụng file concept ontology có sẵn tại: concepts/ntu_rgbd_60_120.csv


In [3]:
# --- 2. Embedded Feeder Tools (Augmentations & Cropping) ---
import random
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        bias = int((1 - p) * valid_size / 2)
        c_b = min(begin + bias, T - 1)
        c_e = max(c_b + 1, min(end - bias, T))
        data = data_numpy[:, c_b:c_e, :, :]  # center_crop
        cropped_length = data.shape[1]
    else:
        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0
        c_b = begin + bias
        c_e = min(begin + bias + cropped_length, T)
        data = data_numpy[:, c_b:c_e, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t
    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)
    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)
        bias = int((1 - p) * valid_size / 2)
        bias = min(bias, max(0, valid_size - cropped_length))

        if cropped_length < window:
            inds = np.arange(cropped_length)
        else:
            bids = np.array(
                [i * cropped_length // window for i in range(window + 1)])
            bst = bids[:window]
            inds = bst

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]

    else:
        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),
                                    valid_size)  # constraint cropped_length lower bound as 64
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0

        if cropped_length < window:
            inds = np.arange(cropped_length)
        elif window <= cropped_length < 2 * window:
            basic = np.arange(window)
            inds = np.random.choice(window + 1, cropped_length - window, replace=False)
            offset = np.zeros(window + 1, dtype=np.int64)
            offset[inds] = 1
            offset = np.cumsum(offset)
            inds = basic + offset[:-1]
        else:
            bids = np.array([i * cropped_length // window for i in range(window + 1)])
            bsize = np.diff(bids)
            bst = bids[:window]
            offset = np.random.randint(bsize)
            inds = bst + offset

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)
    index_t = torch.tensor(inds, dtype=torch.float)
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t

    if len(inds) != window:
        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()

    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def scale(data_numpy, scale=0.2, p=0.5):
    if random.random() < p:
        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)
        return data_numpy * scale
    else:
        return data_numpy.copy()


''' AimCLR '''
transform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}


def subtract(data_numpy, p=0.5):
    joint = random.randint(0, 24)

    C, T, V, M = data_numpy.shape
    if random.random() < p:
        data_numpy_new = np.zeros((C, T, V, M))
        for i in range(V):
            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]
        return data_numpy_new
    else:
        return data_numpy.copy()


def temporal_flip(data_numpy, index_t, p=0.5):
    C, T, V, M = data_numpy.shape
    if random.random() < p:
        time_range_order = [i for i in range(T)]
        time_range_reverse = list(reversed(time_range_order))
        return data_numpy[:, time_range_reverse, :, :], -index_t
    else:
        return data_numpy.copy(), index_t.copy()


def spatial_flip(data_numpy, p=0.5):
    if random.random() < p:
        index = transform_order['ntu']
        return data_numpy[:, :, index, :]
    else:
        return data_numpy.copy()


def rotate(data_numpy, axis=None, angle=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if angle != None:
        angle_next = random.uniform(-angle, angle)
    else:
        angle_next = random.uniform(-30, 30)

    if random.random() < p:
        temp = data_numpy.copy()
        angle = math.radians(angle_next)
        # x
        if axis_next == 0:
            R = np.array([[1, 0, 0],
                          [0, math.cos(angle), math.sin(angle)],
                          [0, -math.sin(angle), math.cos(angle)]])
        # y
        if axis_next == 1:
            R = np.array([[math.cos(angle), 0, -math.sin(angle)],
                          [0, 1, 0],
                          [math.sin(angle), 0, math.cos(angle)]])
        # z
        if axis_next == 2:
            R = np.array([[math.cos(angle), math.sin(angle), 0],
                          [-math.sin(angle), math.cos(angle), 0],
                          [0, 0, 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def shear(data_numpy, s1=None, s2=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        if s1 != None:
            s1_list = s1
        else:
            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]
        if s2 != None:
            s2_list = s2
        else:
            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]

        R = np.array([[1, s1_list[0], s2_list[0]],
                      [s1_list[1], 1, s2_list[1]],
                      [s1_list[2], s2_list[2], 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def drop_axis(data_numpy, axis=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        x_new = np.zeros((T, V, M))
        temp[axis_next] = x_new
        return temp
    else:
        return data_numpy.copy()


def drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape

        if joint_list != None:
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, joint_list)
            joint_list_ = sorted(joint_list_)
        else:
            random_int = random.randint(5, 15)
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, random_int)
            joint_list_ = sorted(joint_list_)

        if time_range != None:
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, time_range)
            time_range_ = sorted(time_range_)
        else:
            random_int = random.randint(16, 32)
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, random_int)
            time_range_ = sorted(time_range_)

        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))
        temp2 = temp[:, time_range_, :, :].copy()
        temp2[:, :, joint_list_, :] = x_new
        temp[:, time_range_, :, :] = temp2
        return temp
    else:
        return data_numpy.copy()


def gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        noise = np.random.normal(mean, std, size=(C, T, V, M))
        return temp + noise
    else:
        return data_numpy.copy()


class GaussianBlurConv(nn.Module):
    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):
        super(GaussianBlurConv, self).__init__()
        self.channels = channels
        self.kernel = kernel
        self.min_max_sigma = sigma
        radius = int(kernel / 2)
        self.kernel_index = np.arange(-radius, radius + 1)
        self.p = p

    def __call__(self, x):
        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])
        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))
        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)
        # kernel =  kernel.float()
        kernel = kernel.double()
        kernel = kernel.repeat(self.channels, 1, 1, 1)
        self.weight = nn.Parameter(data=kernel, requires_grad=False)

        prob = np.random.random_sample()
        x = torch.from_numpy(x).double()
        if prob < self.p:
            x = x.permute(3, 0, 2, 1)
            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)
            x = x.permute(1, -1, -2, 0)

        return x.numpy()


def gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):
    g = GaussianBlurConv(3, kernel, sig_list, p)
    return g(data_numpy)


''' Skeleton AdaIN '''
def skeleton_adain_bone_length(input, ref): # C T V M
    eps = 1e-5
    center = 1
    ref_c = ref[:, :, center, :]

    # joint to bone (joint2bone)
    j2b = joint2bone()
    bone_i = j2b(input) # C T V M
    bone_r = j2b(ref)

    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M
    bone_length_r = np.linalg.norm(bone_r, axis=0)

    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M
    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M

    bone_i = bone_i * bone_length_scale

    # bone to joint (bone2joint)
    b2j = bone2joint()
    joint = b2j(bone_i, ref_c)
    return joint


class joint2bone(nn.Module):
    def __init__(self):
        super(joint2bone, self).__init__()
        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),
                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),
                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, joint):
        bone = np.zeros_like(joint)
        for v1, v2 in self.pairs:
            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]
        return bone


class bone2joint(nn.Module):
    def __init__(self):
        super(bone2joint, self).__init__()
        self.center = 1
        self.pairs_1 = [(0, 1), (20, 1)]
        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]
        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]
        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]
        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]
        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, bone, center):
        joint = np.zeros_like(bone)
        joint[:, :, self.center, :] = center
        for v1, v2 in self.pairs_1:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_2:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_3:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_4:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_5:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_6:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        return joint

def to_motion(input): # C T V M
    C, T, V, M = input.shape
    motion = np.zeros_like(input)
    motion[:, :T - 1] = np.diff(input, axis=1)
    return motion

# The original feeder imports `feeders.tools`.  In this self-contained
# notebook the source is embedded in the global scope, so expose the same
# module-style API expected by Feeder.__getitem__.
tools = types.SimpleNamespace(
    valid_crop_resize=valid_crop_resize,
    valid_crop_uniform=valid_crop_uniform,
    scale=scale,
    subtract=subtract,
    temporal_flip=temporal_flip,
    spatial_flip=spatial_flip,
    rotate=rotate,
    shear=shear,
    drop_axis=drop_axis,
    drop_joint=drop_joint,
    gaussian_noise=gaussian_noise,
    GaussianBlurConv=GaussianBlurConv,
    gaussian_filter=gaussian_filter,
    skeleton_adain_bone_length=skeleton_adain_bone_length,
    joint2bone=joint2bone,
    bone2joint=bone2joint,
    to_motion=to_motion,
)
print("✅ Embedded feeder tools exposed as `tools` namespace!")



✅ Embedded feeder tools exposed as `tools` namespace!


In [4]:
# --- 3. Embedded Feeder Class ---
import numpy as np
import random
import csv

from torch.utils.data import Dataset
# `tools` is exposed as a namespace after the embedded tools source below


class Feeder(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', data_type='j',
                 aug_method='z', intra_p=0.5, inter_p=0.0, window_size=-1,
                 debug=False, thres=64, uniform=False, partition=False,
                 concepts_csv=None, remove_null_concept=True):

        self.debug = debug
        self.data_path = data_path
        self.label_path = label_path
        self.split = split
        self.data_type = data_type
        self.aug_method = aug_method
        self.intra_p = intra_p
        self.inter_p = inter_p
        self.window_size = window_size
        self.p_interval = p_interval
        self.thres = thres
        self.uniform = uniform
        self.partition = partition
        self.concepts_csv = concepts_csv
        self.remove_null_concept = remove_null_concept
        self.load_data()
        self.concept_matrix = None
        self.concept_names = None
        if self.concepts_csv:
            self.load_concepts()
        if partition:
            self.right_arm = np.array([7, 8, 22, 23]) - 1
            self.left_arm = np.array([11, 12, 24, 25]) - 1
            self.right_leg = np.array([13, 14, 15, 16]) - 1
            self.left_leg = np.array([17, 18, 19, 20]) - 1
            self.h_torso = np.array([5, 9, 6, 10]) - 1
            self.w_torso = np.array([2, 3, 1, 4]) - 1
            self.new_idx = np.concatenate((self.right_arm, self.left_arm, self.right_leg, self.left_leg, self.h_torso, self.w_torso), axis=-1)
            # except for joint no.21

    def load_concepts(self):
        """Load REASON concept targets without changing the legacy feeder API."""
        with open(self.concepts_csv, 'r', newline='', encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))

        if not rows or 'action_class' not in rows[0]:
            raise ValueError(f'Invalid concepts CSV: {self.concepts_csv}')

        names = [name for name in rows[0].keys() if name != 'action_class']
        # Keep the same 72-label ordering used by REASON.  These are the two
        # concepts removed by its feeder before concept/logical supervision.
        if self.remove_null_concept:
            names = [name for i, name in enumerate(names) if i not in (50, 65)]

        values = []
        for row in rows:
            values.append([float(row[name]) for name in names])
        self.concept_names = names
        self.concept_matrix = np.asarray(values, dtype=np.float32)
        if self.concept_matrix.shape[1] != 72:
            raise ValueError(
                f'Expected 72 concepts after filtering, got {self.concept_matrix.shape[1]}'
            )

    def load_data(self):
        # data: N C V T M
        npz_data = np.load(self.data_path)
        if self.split == 'train':
            self.data = npz_data['x_train']
            self.label = np.where(npz_data['y_train'] > 0)[1]
            self.sample_name = ['train_' + str(i) for i in range(len(self.data))]
        elif self.split == 'test':
            self.data = npz_data['x_test']
            self.label = np.where(npz_data['y_test'] > 0)[1]
            self.sample_name = ['test_' + str(i) for i in range(len(self.data))]
        else:
            raise NotImplementedError('data split only supports train/test')
        N, T, _ = self.data.shape
        self.data = self.data.reshape((N, T, 2, 25, 3)).transpose(0, 4, 1, 3, 2)

    def __len__(self):
        return len(self.label)

    def __iter__(self):
        return self

    def __getitem__(self, index):
        data_numpy = self.data[index]
        label = self.label[index]
        data_numpy = np.array(data_numpy)
        valid_frame_num = np.sum(data_numpy.sum(0).sum(-1).sum(-1) != 0)
        num_people = np.sum(data_numpy.sum(0).sum(0).sum(0) != 0)

        if self.uniform:
            data_numpy, index_t = tools.valid_crop_uniform(data_numpy, valid_frame_num, self.p_interval,
                                                           self.window_size, self.thres)
        else:
            data_numpy, index_t = tools.valid_crop_resize(data_numpy, valid_frame_num, self.p_interval,
                                                          self.window_size, self.thres)

        if self.split == 'train':
            # intra-instance augmentation
            p = np.random.rand(1)
            if p < self.intra_p:

                if 'a' in self.aug_method:
                    if np.random.rand(1) < 0.5:
                        data_numpy = data_numpy[:, :, :, np.array([1, 0])]
                if 'b' in self.aug_method:
                    if num_people == 2:
                        if np.random.rand(1) < 0.5:
                            axis_next = np.random.randint(0, 1)
                            temp = data_numpy.copy()
                            C, T, V, M = data_numpy.shape
                            x_new = np.zeros((C, T, V))
                            temp[:, :, :, axis_next] = x_new
                            data_numpy = temp

                if '1' in self.aug_method:
                    data_numpy = tools.shear(data_numpy, p=0.5)
                if '2' in self.aug_method:
                    data_numpy = tools.rotate(data_numpy, p=0.5)
                if '3' in self.aug_method:
                    data_numpy = tools.scale(data_numpy, p=0.5)
                if '4' in self.aug_method:
                    data_numpy = tools.spatial_flip(data_numpy, p=0.5)
                if '5' in self.aug_method:
                    data_numpy, index_t = tools.temporal_flip(data_numpy, index_t, p=0.5)
                if '6' in self.aug_method:
                    data_numpy = tools.gaussian_noise(data_numpy, p=0.5)
                if '7' in self.aug_method:
                    data_numpy = tools.gaussian_filter(data_numpy, p=0.5)
                if '8' in self.aug_method:
                    data_numpy = tools.drop_axis(data_numpy, p=0.5)
                if '9' in self.aug_method:
                    data_numpy = tools.drop_joint(data_numpy, p=0.5)

            # inter-instance augmentation
            elif (p < (self.intra_p + self.inter_p)) & (p >= self.intra_p):
                adain_idx = random.choice(np.where(self.label == label)[0])
                data_adain = self.data[adain_idx]
                data_adain = np.array(data_adain)
                f_num = np.sum(data_adain.sum(0).sum(-1).sum(-1) != 0)
                t_idx = np.round((index_t + 1) * f_num / 2).astype(np.int64)
                data_adain = data_adain[:, t_idx]
                data_numpy = tools.skeleton_adain_bone_length(data_numpy, data_adain)

            else:
                data_numpy = data_numpy.copy()

        # modality
        if self.data_type == 'b':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
        elif self.data_type == 'jm':
            data_numpy = tools.to_motion(data_numpy)
        elif self.data_type == 'bm':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
            data_numpy = tools.to_motion(data_numpy)
        else:
            data_numpy = data_numpy.copy()

        if self.partition:
            data_numpy = data_numpy[:, :, self.new_idx]

        if self.concept_matrix is not None:
            if label >= len(self.concept_matrix):
                raise IndexError(f'Concept CSV has no row for action label {label}')
            return data_numpy, index_t, label, index, self.concept_matrix[label]
        return data_numpy, index_t, label, index

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) * 1.0 / len(hit_top_k)


def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod


# Đăng ký alias vào sys.modules để hỗ trợ mọi lệnh import legacy nếu có
_feeders_mod = types.ModuleType("feeders")
_feeders_feeder_mod = types.ModuleType("feeders.feeder_ntu")
_feeders_tools_mod = types.ModuleType("feeders.tools")
_feeders_feeder_mod.Feeder = Feeder
sys.modules["feeders"] = _feeders_mod
sys.modules["feeders.feeder_ntu"] = _feeders_feeder_mod
sys.modules["feeders.tools"] = _feeders_tools_mod
print("✅ Feeder & Data Augmentations loaded successfully!")



✅ Feeder & Data Augmentations loaded successfully!


## 🧠 Block 3: Kiến trúc SkateFormer Backbone


In [5]:
# --- Embedded source: model/SkateFormer.py ---
from typing import Type, Tuple, Optional, Set, List, Union

import math
import numpy as np
import torch
import torch.nn as nn
from torch.nn.init import trunc_normal_


def drop_path(x, drop_prob=0.0, training=False):
    """DropPath implementation compatible with timm's stochastic depth."""
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class Mlp(nn.Module):
    """Minimal timm-compatible MLP used by SkateFormer."""
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        out_features = out_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x

''' Partition and Reverse '''


def type_1_partition(input, partition_size):  # partition_size = [N, L]
    B, C, T, V = input.shape
    partitions = input.view(B, C, T // partition_size[0], partition_size[0], V // partition_size[1], partition_size[1])
    partitions = partitions.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_1_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, -1, T, V)
    return output


def type_2_partition(input, partition_size):  # partition_size = [N, K]
    B, C, T, V = input.shape
    partitions = input.view(B, C, T // partition_size[0], partition_size[0], partition_size[1], V // partition_size[1])
    partitions = partitions.permute(0, 2, 5, 3, 4, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_2_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 1, 3, 4, 2).contiguous().view(B, -1, T, V)
    return output


def type_3_partition(input, partition_size):  # partition_size = [M, L]
    B, C, T, V = input.shape
    partitions = input.view(B, C, partition_size[0], T // partition_size[0], V // partition_size[1], partition_size[1])
    partitions = partitions.permute(0, 3, 4, 2, 5, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_3_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 3, 1, 2, 4).contiguous().view(B, -1, T, V)
    return output


def type_4_partition(input, partition_size):  # partition_size = [M, K]
    B, C, T, V = input.shape
    partitions = input.view(B, C, partition_size[0], T // partition_size[0], partition_size[1], V // partition_size[1])
    partitions = partitions.permute(0, 3, 5, 2, 4, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_4_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 3, 1, 4, 2).contiguous().view(B, -1, T, V)
    return output


''' 1D relative positional bias: B_{h}^{t} '''


def get_relative_position_index_1d(T):
    coords = torch.stack(torch.meshgrid([torch.arange(T)]))
    coords_flatten = torch.flatten(coords, 1)
    relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()
    relative_coords[:, :, 0] += T - 1
    return relative_coords.sum(-1)


''' MSA '''


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, in_channels, rel_type, num_heads=32, partition_size=(1, 1), attn_drop=0., rel=True):
        super(MultiHeadSelfAttention, self).__init__()
        self.in_channels = in_channels
        self.rel_type = rel_type
        self.num_heads = num_heads
        self.partition_size = partition_size
        self.scale = num_heads ** -0.5
        self.attn_area = partition_size[0] * partition_size[1]
        self.attn_drop = nn.Dropout(p=attn_drop)
        self.softmax = nn.Softmax(dim=-1)
        self.rel = rel

        if self.rel:
            if self.rel_type == 'type_1' or self.rel_type == 'type_3':
                self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * partition_size[0] - 1), num_heads))
                self.register_buffer("relative_position_index", get_relative_position_index_1d(partition_size[0]))
                trunc_normal_(self.relative_position_bias_table, std=.02)
                self.ones = torch.ones(partition_size[1], partition_size[1], num_heads)
            elif self.rel_type == 'type_2' or self.rel_type == 'type_4':
                self.relative_position_bias_table = nn.Parameter(
                    torch.zeros((2 * partition_size[0] - 1), partition_size[1], partition_size[1], num_heads))
                self.register_buffer("relative_position_index", get_relative_position_index_1d(partition_size[0]))
                trunc_normal_(self.relative_position_bias_table, std=.02)

    def _get_relative_positional_bias(self):
        if self.rel_type == 'type_1' or self.rel_type == 'type_3':
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
                self.partition_size[0], self.partition_size[0], -1)
            relative_position_bias = relative_position_bias.unsqueeze(1).unsqueeze(3).repeat(1, self.partition_size[1], 1, self.partition_size[1], 1, 1).view(self.attn_area, self.attn_area, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            return relative_position_bias.unsqueeze(0)
        elif self.rel_type == 'type_2' or self.rel_type == 'type_4':
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(self.partition_size[0], self.partition_size[0], self.partition_size[1], self.partition_size[1], -1)
            relative_position_bias = relative_position_bias.permute(0, 2, 1, 3, 4).contiguous().view(self.attn_area, self.attn_area, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            return relative_position_bias.unsqueeze(0)

    def forward(self, input):
        B_, N, C = input.shape
        qkv = input.reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if self.rel:
            attn = attn + self._get_relative_positional_bias()
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        output = (attn @ v).transpose(1, 2).reshape(B_, N, -1)
        return output


''' SkateFormer Block '''


class SkateFormerBlock(nn.Module):
    def __init__(self, in_channels, num_points=50, kernel_size=7, num_heads=32,
                 type_1_size=(1, 1), type_2_size=(1, 1), type_3_size=(1, 1), type_4_size=(1, 1),
                 attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super(SkateFormerBlock, self).__init__()
        self.type_1_size = type_1_size
        self.type_2_size = type_2_size
        self.type_3_size = type_3_size
        self.type_4_size = type_4_size
        self.partition_function = [type_1_partition, type_2_partition, type_3_partition, type_4_partition]
        self.reverse_function = [type_1_reverse, type_2_reverse, type_3_reverse, type_4_reverse]
        self.partition_size = [type_1_size, type_2_size, type_3_size, type_4_size]
        self.rel_type = ['type_1', 'type_2', 'type_3', 'type_4']

        self.norm_1 = norm_layer(in_channels)
        self.mapping = nn.Linear(in_features=in_channels, out_features=2 * in_channels, bias=True)
        self.gconv = nn.Parameter(torch.zeros(num_heads // (2 * 2), num_points, num_points))
        trunc_normal_(self.gconv, std=.02)
        self.tconv = nn.Conv2d(in_channels // (2 * 2), in_channels // (2 * 2), kernel_size=(kernel_size, 1),
                               padding=((kernel_size - 1) // 2, 0), groups=num_heads // (2 * 2))

        # Attention layers
        attention = []
        for i in range(len(self.partition_function)):
            attention.append(
                MultiHeadSelfAttention(in_channels=in_channels // (len(self.partition_function) * 2),
                                       rel_type=self.rel_type[i],
                                       num_heads=num_heads // (len(self.partition_function) * 2),
                                       partition_size=self.partition_size[i], attn_drop=attn_drop, rel=rel))
        self.attention = nn.ModuleList(attention)
        self.proj = nn.Linear(in_features=in_channels, out_features=in_channels, bias=True)
        self.proj_drop = nn.Dropout(p=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm_2 = norm_layer(in_channels)
        self.mlp = Mlp(in_features=in_channels, hidden_features=int(mlp_ratio * in_channels),
                       act_layer=act_layer, drop=drop)

    def forward(self, input):
        B, C, T, V = input.shape

        # Partition
        input = input.permute(0, 2, 3, 1).contiguous()
        skip = input

        f = self.mapping(self.norm_1(input)).permute(0, 3, 1, 2).contiguous()

        f_conv, f_attn = torch.split(f, [C // 2, 3 * C // 2], dim=1)
        y = []

        # G-Conv
        split_f_conv = torch.chunk(f_conv, 2, dim=1)
        y_gconv = []
        split_f_gconv = torch.chunk(split_f_conv[0], self.gconv.shape[0], dim=1)
        for i in range(self.gconv.shape[0]):
            z = torch.einsum('n c t u, v u -> n c t v', split_f_gconv[i], self.gconv[i])
            y_gconv.append(z)
        y.append(torch.cat(y_gconv, dim=1))  # N C T V

        # T-Conv
        y.append(self.tconv(split_f_conv[1]))

        # Skate-MSA
        split_f_attn = torch.chunk(f_attn, len(self.partition_function), dim=1)

        for i in range(len(self.partition_function)):
            C = split_f_attn[i].shape[1]
            input_partitioned = self.partition_function[i](split_f_attn[i], self.partition_size[i])
            input_partitioned = input_partitioned.view(-1, self.partition_size[i][0] * self.partition_size[i][1], C)
            y.append(self.reverse_function[i](self.attention[i](input_partitioned), (T, V), self.partition_size[i]))

        output = self.proj(torch.cat(y, dim=1).permute(0, 2, 3, 1).contiguous())
        output = self.proj_drop(output)
        output = skip + self.drop_path(output)

        # Feed Forward
        output = output + self.drop_path(self.mlp(self.norm_2(output)))
        output = output.permute(0, 3, 1, 2).contiguous()
        return output


''' Downsampling '''


class PatchMergingTconv(nn.Module):
    def __init__(self, dim_in, dim_out, kernel_size=7, stride=2, dilation=1):
        super().__init__()
        self.dim_in = dim_in
        self.dim_out = dim_out
        pad = (kernel_size + (kernel_size - 1) * (dilation - 1) - 1) // 2
        self.reduction = nn.Conv2d(dim_in, dim_out, kernel_size=(kernel_size, 1), padding=(pad, 0), stride=(stride, 1),
                                   dilation=(dilation, 1))
        self.bn = nn.BatchNorm2d(dim_out)

    def forward(self, x):
        x = self.bn(self.reduction(x))
        return x


''' SkateFormer Block with Downsampling '''


class SkateFormerBlockDS(nn.Module):
    def __init__(
            self, in_channels, out_channels, num_points=50, kernel_size=7, downscale=False, num_heads=32,
            type_1_size=(1, 1), type_2_size=(1, 1), type_3_size=(1, 1), type_4_size=(1, 1),
            attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
            act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm):
        super(SkateFormerBlockDS, self).__init__()

        if downscale:
            self.downsample = PatchMergingTconv(in_channels, out_channels, kernel_size=kernel_size)
        else:
            self.downsample = None

        self.transformer = SkateFormerBlock(
            in_channels=out_channels,
            num_points=num_points,
            kernel_size=kernel_size,
            num_heads=num_heads,
            type_1_size=type_1_size,
            type_2_size=type_2_size,
            type_3_size=type_3_size,
            type_4_size=type_4_size,
            attn_drop=attn_drop,
            drop=drop,
            rel=rel,
            drop_path=drop_path,
            mlp_ratio=mlp_ratio,
            act_layer=act_layer,
            norm_layer=norm_layer_transformer,
        )

    def forward(self, input):
        if self.downsample is not None:
            output = self.transformer(self.downsample(input))
        else:
            output = self.transformer(input)
        return output


''' SkateFormer Stage '''


class SkateFormerStage(nn.Module):
    def __init__(
            self, depth, in_channels, out_channels, first_depth=False,
            num_points=50, kernel_size=7, num_heads=32,
            type_1_size=(1, 1), type_2_size=(1, 1), type_3_size=(1, 1), type_4_size=(1, 1),
            attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
            act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm):
        super(SkateFormerStage, self).__init__()
        blocks = []
        for index in range(depth):
            blocks.append(
                SkateFormerBlockDS(
                    in_channels=in_channels if index == 0 else out_channels,
                    out_channels=out_channels,
                    num_points=num_points,
                    kernel_size=kernel_size,
                    downscale=((index == 0) & ~first_depth),
                    num_heads=num_heads,
                    type_1_size=type_1_size,
                    type_2_size=type_2_size,
                    type_3_size=type_3_size,
                    type_4_size=type_4_size,
                    attn_drop=attn_drop,
                    drop=drop,
                    rel=rel,
                    drop_path=drop_path if isinstance(drop_path, float) else drop_path[index],
                    mlp_ratio=mlp_ratio,
                    act_layer=act_layer,
                    norm_layer_transformer=norm_layer_transformer))
        self.blocks = nn.ModuleList(blocks)

    def forward(self, input):
        output = input
        for block in self.blocks:
            output = block(output)
        return output


''' SkateFormer '''


class SkateFormer(nn.Module):
    def __init__(self, in_channels=3, depths=(2, 2, 2, 2), channels=(96, 192, 192, 192), num_classes=60,
                 embed_dim=64, num_people=2, num_frames=64, num_points=50, kernel_size=7, num_heads=32,
                 type_1_size=(1, 1), type_2_size=(1, 1), type_3_size=(1, 1), type_4_size=(1, 1),
                 attn_drop=0., head_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
                 act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm, index_t=False, global_pool='avg'):

        super(SkateFormer, self).__init__()

        assert len(depths) == len(channels), "For each stage a channel dimension must be given."
        assert global_pool in ["avg", "max"], f"Only avg and max is supported but {global_pool} is given"
        self.num_classes: int = num_classes
        self.head_drop = head_drop
        self.index_t = index_t
        self.embed_dim = embed_dim

        if self.head_drop != 0:
            self.dropout = nn.Dropout(p=self.head_drop)
        else:
            self.dropout = None

        stem = []
        stem.append(nn.Conv2d(in_channels=in_channels, out_channels=2 * in_channels, kernel_size=(1, 1), stride=(1, 1),
                              padding=(0, 0)))
        stem.append(act_layer())
        stem.append(
            nn.Conv2d(in_channels=2 * in_channels, out_channels=3 * in_channels, kernel_size=(1, 1), stride=(1, 1),
                      padding=(0, 0)))
        stem.append(act_layer())
        stem.append(nn.Conv2d(in_channels=3 * in_channels, out_channels=embed_dim, kernel_size=(1, 1), stride=(1, 1),
                              padding=(0, 0)))
        self.stem = nn.ModuleList(stem)

        if self.index_t:
            self.joint_person_embedding = nn.Parameter(torch.zeros(embed_dim, num_points * num_people))
            trunc_normal_(self.joint_person_embedding, std=.02)
        else:
            self.joint_person_temporal_embedding = nn.Parameter(
                torch.zeros(1, embed_dim, num_frames, num_points * num_people))
            trunc_normal_(self.joint_person_temporal_embedding, std=.02)

        # Init blocks
        drop_path = torch.linspace(0.0, drop_path, sum(depths)).tolist()
        stages = []
        for index, (depth, channel) in enumerate(zip(depths, channels)):
            stages.append(
                SkateFormerStage(
                    depth=depth,
                    in_channels=embed_dim if index == 0 else channels[index - 1],
                    out_channels=channel,
                    first_depth=index == 0,
                    num_points=num_points * num_people,
                    kernel_size=kernel_size,
                    num_heads=num_heads,
                    type_1_size=type_1_size,
                    type_2_size=type_2_size,
                    type_3_size=type_3_size,
                    type_4_size=type_4_size,
                    attn_drop=attn_drop,
                    drop=drop,
                    rel=rel,
                    drop_path=drop_path[sum(depths[:index]):sum(depths[:index + 1])],
                    mlp_ratio=mlp_ratio,
                    act_layer=act_layer,
                    norm_layer_transformer=norm_layer_transformer
                )
            )
        self.stages = nn.ModuleList(stages)
        self.global_pool: str = global_pool
        self.head = nn.Linear(channels[-1], num_classes)

    @torch.jit.ignore
    def no_weight_decay(self):
        nwd = set()
        for n, _ in self.named_parameters():
            if "relative_position_bias_table" in n:
                nwd.add(n)
        return nwd

    def reset_classifier(self, num_classes, global_pool=None):
        self.num_classes: int = num_classes
        if global_pool is not None:
            self.global_pool = global_pool
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, input):
        output = input
        for stage in self.stages:
            output = stage(output)
        return output

    def forward_head(self, input, pre_logits=False):
        if self.global_pool == "avg":
            input = input.mean(dim=(2, 3))
        elif self.global_pool == "max":
            input = torch.amax(input, dim=(2, 3))
        if self.dropout is not None:
            input = self.dropout(input)
        return input if pre_logits else self.head(input)

    def forward_features_from_input(self, input, index_t=None):
        """Return the final feature map before global pooling/classification.

        This is shared by the original classifier and REASON-SkateFormer so
        the pretrained SkateFormer stem and temporal embedding are identical
        in both paths.
        """
        B, C, T, V, M = input.shape

        output = input.permute(0, 1, 2, 4, 3).contiguous().view(B, C, T, -1)  # [B, C, T, M * V]
        for layer in self.stem:
            output = layer(output)
        if self.index_t:
            if index_t is None:
                index_t = torch.linspace(-1., 1., T, device=output.device, dtype=output.dtype)
                index_t = index_t.unsqueeze(0).expand(B, T)
            te = torch.zeros(B, T, self.embed_dim).to(output.device)  # B, T, C
            div_term = torch.exp(
                (torch.arange(0, self.embed_dim, 2, dtype=torch.float) * -(math.log(10000.0) / self.embed_dim))).to(
                output.device)
            te[:, :, 0::2] = torch.sin(index_t.unsqueeze(-1).float() * div_term)
            te[:, :, 1::2] = torch.cos(index_t.unsqueeze(-1).float() * div_term)
            output = output + torch.einsum('b t c, c v -> b c t v', te, self.joint_person_embedding)
        else:
            output = output + self.joint_person_temporal_embedding
        return self.forward_features(output)

    def forward(self, input, index_t=None):
        output = self.forward_features_from_input(input, index_t=index_t)
        output = self.forward_head(output)
        return output


def SkateFormer_(**kwargs):
    return SkateFormer(
        depths=(2, 2, 2, 2),
        channels=(96, 192, 192, 192),
        embed_dim=96,
        **kwargs
    )


_skate_mod = types.ModuleType("model.SkateFormer")
_skate_mod.SkateFormer_ = SkateFormer_
sys.modules["model.SkateFormer"] = _skate_mod
print("✅ SkateFormer backbone loaded successfully!")



✅ SkateFormer backbone loaded successfully!


## 🧠 Block 4: Module Looping Concept Reasoning (LCR-SkateFormer)


In [6]:
# --- Embedded source: model/lcr_skateformer.py ---
"""LCR-SkateFormer: Looping Concept Reasoning SkateFormer.

SkateFormer serves as the perception backbone to extract initial concept logits (z^0).
An iterative knowledge-graph-guided reasoning engine (IterativeKGReasoner) then refines
the concept representations over K loops (z^t -> z^{t+1} = z^t + beta * Delta z^t)
in concept/logit space, generating progressive action evidence and verifiable reasoning traces.
"""

from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# 'SkateFormer_' is embedded directly in notebook scope


def _logit(probability: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    """Stable logit function mapping probabilities in (0, 1) to (-inf, +inf)."""
    probability = probability.clamp(eps, 1.0 - eps)
    return torch.log(probability) - torch.log1p(-probability)


class KGUnionLayer(nn.Module):
    """Differentiable conjunction (AND) / disjunction (OR) rule bank.

    Weights W are stored in logit space and constrained to [0, 1] via sigmoid.
    The rule bank can be initialized with domain-specific Knowledge Graph priors.
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        prior: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

        if prior is None:
            prior = torch.full((input_dim, output_dim), 0.05)
            diag_len = min(input_dim, output_dim)
            prior[torch.arange(diag_len), torch.arange(diag_len)] = 0.9

        prior = prior.float().clamp(1e-4, 1.0 - 1e-4)
        self.register_buffer("prior", prior.clone())

        noise = 0.02 * torch.randn_like(prior)
        self.and_weight = nn.Parameter(_logit((prior + noise).clamp(1e-4, 1.0 - 1e-4)))
        self.or_weight = nn.Parameter(_logit((prior + noise).clamp(1e-4, 1.0 - 1e-4)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass applying AND and OR soft operations.

        Args:
            x: Input literals [B, input_dim] in [0, 1].

        Returns:
            Concatenated [conjunction, disjunction] tensor of shape [B, 2 * output_dim].
        """
        and_weight = torch.sigmoid(self.and_weight)  # [input_dim, output_dim]
        or_weight = torch.sigmoid(self.or_weight)    # [input_dim, output_dim]

        # Soft conjunction (AND): prod(1 - w_i * (1 - x_i))
        conjunction = torch.prod(
            1.0 - and_weight.unsqueeze(0) * (1.0 - x.unsqueeze(-1)), dim=1
        )
        # Soft disjunction (OR): 1 - prod(1 - w_i * x_i)
        disjunction = 1.0 - torch.prod(
            1.0 - or_weight.unsqueeze(0) * x.unsqueeze(-1), dim=1
        )
        return torch.cat([conjunction, disjunction], dim=1)

    def weights(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return the bounded AND/OR rule weights."""
        return torch.sigmoid(self.and_weight), torch.sigmoid(self.or_weight)

    def prior_retention_loss(self) -> torch.Tensor:
        """Discourage learned rule weights from drifting too far from KG initialization."""
        and_weight, or_weight = self.weights()
        return 0.5 * (
            (and_weight - self.prior).abs().mean()
            + (or_weight - self.prior).abs().mean()
        )

    def sparsity_loss(self) -> torch.Tensor:
        """Penalize redundant active rule connections."""
        and_weight, or_weight = self.weights()
        return 0.5 * (and_weight.mean() + or_weight.mean())


class KGPrimedReasoner(nn.Module):
    """Concept-to-action logical reasoner with KG-primed rule banks.

    Supports both 'linear' action classification and 'prototype' schema compatibility.
    """

    def __init__(
        self,
        n_concepts: int,
        n_actions: int,
        concept_names: List[str],
        l1_dim: int = 128,
        l2_dim: int = 128,
        kg_init_mode: str = "kg_primed",
        kg_prior_map: Optional[Dict[str, List]] = None,
        kg_prior_strength: float = 1.0,
        action_mode: str = "linear",
        action_prototypes: Optional[Union[torch.Tensor, List]] = None,
        prototype_and_weight: float = 0.35,
    ):
        super().__init__()
        self.n_concepts = int(n_concepts)
        self.n_actions = int(n_actions)
        self.concept_names = list(concept_names)
        if len(self.concept_names) != self.n_concepts:
            raise ValueError(
                f"Expected {self.n_concepts} concept names, got {len(self.concept_names)}"
            )

        self.kg_init_mode = str(kg_init_mode).lower()
        self.kg_prior_map = kg_prior_map or {"require": [], "alongside": [], "excludes": []}
        self.kg_prior_strength = float(kg_prior_strength)
        self.action_mode = str(action_mode).lower()
        self.prototype_and_weight = float(prototype_and_weight)
        self.l1_dim = int(l1_dim)
        self.l2_dim = int(l2_dim)

        if self.action_mode == "prototype":
            if action_prototypes is None:
                raise ValueError("action_prototypes are required when action_mode='prototype'")
            prototypes = torch.as_tensor(action_prototypes, dtype=torch.float32)
            if prototypes.shape[0] > self.n_actions:
                prototypes = prototypes[: self.n_actions]
            if prototypes.shape != (self.n_actions, self.n_concepts):
                raise ValueError(
                    f"action_prototypes shape must be ({self.n_actions}, {self.n_concepts}), got {tuple(prototypes.shape)}"
                )
            prototypes = (prototypes >= 0.5).float()
            self.register_buffer("action_prototypes", prototypes)
            action_prior = self._build_action_prior(prototypes)
            self.action_rule = KGUnionLayer(
                2 * self.n_concepts, self.n_actions, prior=action_prior
            )
            self.logic_dim = 2 * self.n_actions
        else:
            first_prior = None
            if self.kg_init_mode == "kg_primed":
                first_prior = self._build_first_layer_prior(2 * self.n_concepts, self.l1_dim)

            self.first = KGUnionLayer(2 * self.n_concepts, self.l1_dim, prior=first_prior)
            self.second = KGUnionLayer(2 * self.l1_dim, self.l2_dim)
            self.action = nn.Linear(2 * self.l2_dim, self.n_actions)
            self.logic_dim = 2 * self.l1_dim + 2 * self.l2_dim

    def _resolve(self, name: Union[str, int]) -> Optional[int]:
        if isinstance(name, int):
            return name if 0 <= name < self.n_concepts else None
        key = str(name).strip().lower()
        for idx, cname in enumerate(self.concept_names):
            if str(cname).strip().lower() == key:
                return idx
        return None

    def _build_first_layer_prior(self, input_dim: int, output_dim: int) -> torch.Tensor:
        prior = torch.full((input_dim, output_dim), 0.03)
        relations = {
            "require": self.kg_prior_map.get("require", []),
            "alongside": self.kg_prior_map.get("alongside", []),
            "excludes": self.kg_prior_map.get("excludes", []),
        }

        for rule_idx in range(output_dim):
            seed = rule_idx % self.n_concepts
            prior[seed, rule_idx] = 0.95
            for rel_name, pairs in relations.items():
                val = {
                    "require": 0.90,
                    "alongside": 0.70,
                    "excludes": 0.01,
                }[rel_name] * self.kg_prior_strength
                val = max(0.01, min(0.98, val))
                for pair in pairs:
                    if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                        continue
                    a = self._resolve(pair[0])
                    b = self._resolve(pair[1])
                    if a is None or b is None:
                        continue
                    if seed == a:
                        prior[b, rule_idx] = val
                    elif seed == b:
                        prior[a, rule_idx] = val
        return prior

    def _build_action_prior(self, prototypes: torch.Tensor) -> torch.Tensor:
        prior = torch.full(
            (2 * self.n_concepts, self.n_actions), 1e-4, dtype=prototypes.dtype
        )
        prior[: self.n_concepts] = prototypes.t() * 0.95
        for action_idx in range(self.n_actions):
            pos_concepts = set(torch.nonzero(prototypes[action_idx] > 0.5).flatten().tolist())
            for rel_name, pairs in self.kg_prior_map.items():
                if rel_name not in {"require", "alongside", "excludes"}:
                    continue
                rel_val = {
                    "require": 0.90,
                    "alongside": 0.70,
                    "excludes": 0.90,
                }[rel_name] * self.kg_prior_strength
                rel_val = max(0.01, min(0.98, rel_val))
                for pair in pairs:
                    if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                        continue
                    a = self._resolve(pair[0])
                    b = self._resolve(pair[1])
                    if a is None or b is None:
                        continue
                    if rel_name == "require":
                        if b in pos_concepts:
                            prior[a, action_idx] = max(prior[a, action_idx], rel_val)
                    elif rel_name == "alongside":
                        if a in pos_concepts:
                            prior[b, action_idx] = max(prior[b, action_idx], rel_val)
                        if b in pos_concepts:
                            prior[a, action_idx] = max(prior[a, action_idx], rel_val)
                    else:
                        if a in pos_concepts:
                            prior[self.n_concepts + b, action_idx] = max(
                                prior[self.n_concepts + b, action_idx], rel_val
                            )
                        if b in pos_concepts:
                            prior[self.n_concepts + a, action_idx] = max(
                                prior[self.n_concepts + a, action_idx], rel_val
                            )
        return prior.clamp(1e-4, 0.98)

    def kg_loss(self, concept_logits: torch.Tensor) -> torch.Tensor:
        """Soft consistency penalty for domain KG relations."""
        probs = torch.sigmoid(concept_logits)
        penalties = []
        for rel_name in ("require", "alongside", "excludes"):
            for pair in self.kg_prior_map.get(rel_name, []):
                if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                    continue
                a = self._resolve(pair[0])
                b = self._resolve(pair[1])
                if a is None or b is None:
                    continue
                pa, pb = probs[:, a], probs[:, b]
                if rel_name == "require":
                    # pb requires pa: penalty if pb > pa
                    penalties.append(F.relu(pb - pa))
                elif rel_name == "alongside":
                    # Co-occurrence expected: penalty if low joint probability
                    penalties.append(F.relu(0.5 - pa * pb))
                else:
                    # Mutual exclusion: penalty if both are high
                    penalties.append(pa * pb)
        if not penalties:
            return concept_logits.new_zeros(())
        return torch.stack([p.mean() for p in penalties]).mean()

    def rule_prior_loss(self) -> torch.Tensor:
        if self.action_mode == "prototype":
            return self.action_rule.prior_retention_loss()
        return 0.5 * (self.first.prior_retention_loss() + self.second.prior_retention_loss())

    def rule_sparsity_loss(self) -> torch.Tensor:
        if self.action_mode == "prototype":
            return self.action_rule.sparsity_loss()
        return 0.5 * (self.first.sparsity_loss() + self.second.sparsity_loss())

    def forward(
        self,
        concept_logits: torch.Tensor,
        return_trace: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict]:
        """Forward pass of reasoner.

        Args:
            concept_logits: [B, n_concepts]
            return_trace: whether to return intermediate rule tensors

        Returns:
            action_logits: [B, n_actions]
            logic_state: [B, logic_dim]
            trace_dict: dict of rule activations
        """
        concepts = torch.sigmoid(concept_logits)
        literals = torch.cat([concepts, 1.0 - concepts], dim=-1)  # [B, 2 * n_concepts]

        if self.action_mode == "prototype":
            rule_output = self.action_rule(literals)  # [B, 2 * n_actions]
            and_score, or_score = rule_output.chunk(2, dim=-1)
            evidence = (
                self.prototype_and_weight * and_score
                + (1.0 - self.prototype_and_weight) * or_score
            ).clamp(1e-4, 1.0 - 1e-4)
            action_logits = _logit(evidence)
            logic_state = rule_output
            trace = {
                "and_score": and_score,
                "or_score": or_score,
                "evidence": evidence,
                "literals": literals,
            }
        else:
            hidden_1 = self.first(literals)  # [B, 2 * l1_dim]
            hidden_2 = self.second(hidden_1)  # [B, 2 * l2_dim]
            action_logits = self.action(hidden_2)
            logic_state = torch.cat([hidden_1, hidden_2], dim=-1)
            trace = {
                "hidden_1": hidden_1,
                "hidden_2": hidden_2,
                "literals": literals,
            }

        return action_logits, logic_state, trace


class ConceptUpdateCell(nn.Module):
    """Concept Refinement Cell.

    Maps current concept logit state z^t and logical evidence r^t (plus optional action evidence)
    to a residual logit update Delta z^t.
    Notice: Visual feature maps are NOT fed into this cell to strictly protect
    the concept bottleneck from black-box shortcutting.
    """

    def __init__(
        self,
        num_concepts: int,
        logic_dim: int,
        action_dim: Optional[int] = None,
        hidden_dim: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.logic_dim = int(logic_dim)
        self.has_action_feedback = action_dim is not None and action_dim > 0

        # Project logic evidence to concept space
        self.logic_projection = nn.Sequential(
            nn.Linear(logic_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_concepts),
        )

        in_dim = num_concepts * 2
        if self.has_action_feedback:
            self.action_projection = nn.Sequential(
                nn.Linear(action_dim, hidden_dim // 2),
                nn.GELU(),
                nn.Linear(hidden_dim // 2, num_concepts),
            )
            in_dim += num_concepts

        self.update_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_concepts),
        )

    def forward(
        self,
        z_t: torch.Tensor,
        logic_state: torch.Tensor,
        action_logits: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Compute residual logit delta Delta z^t.

        Args:
            z_t: Current concept logits [B, num_concepts]
            logic_state: Rule bank logic evidence [B, logic_dim]
            action_logits: Optional action logits [B, num_actions]

        Returns:
            delta_z: [B, num_concepts]
        """
        r_proj = self.logic_projection(logic_state)
        inputs = [z_t, r_proj]
        if self.has_action_feedback and action_logits is not None:
            act_proj = self.action_projection(action_logits)
            inputs.append(act_proj)
        update_in = torch.cat(inputs, dim=-1)
        return self.update_mlp(update_in)


class IterativeKGReasoner(nn.Module):
    """KG-guided Looping Concept Reasoning Engine (LCR).

    Iterates over K reasoning loops to refine concept logits and generate
    consistent action evidence step-by-step.
    """

    def __init__(
        self,
        num_concepts: int,
        num_actions: int,
        concept_names: List[str],
        num_loops: int = 2,
        shared_weights: bool = True,
        update_gate_init: float = 0.1,
        max_gate: float = 0.5,
        use_action_feedback: bool = False,
        l1_dim: int = 128,
        l2_dim: int = 128,
        kg_init_mode: str = "kg_primed",
        kg_prior_map: Optional[Dict[str, List]] = None,
        kg_prior_strength: float = 1.0,
        action_mode: str = "linear",
        action_prototypes: Optional[Union[torch.Tensor, List]] = None,
        prototype_and_weight: float = 0.35,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.num_actions = int(num_actions)
        self.num_loops = max(1, int(num_loops))
        self.shared_weights = bool(shared_weights)
        self.max_gate = float(max_gate)
        self.use_action_feedback = bool(use_action_feedback)
        self.action_mode = str(action_mode).lower()

        # Learnable gating parameter initialized to yield update_gate_init
        # beta = sigmoid(raw_beta) * max_gate
        init_gate = min(max(update_gate_init, 1e-4), max_gate - 1e-4)
        raw_init = _logit(torch.tensor(init_gate / self.max_gate, dtype=torch.float32))
        self.raw_beta = nn.Parameter(raw_init.clone().detach())

        def _make_reasoner():
            return KGPrimedReasoner(
                n_concepts=self.num_concepts,
                n_actions=self.num_actions,
                concept_names=concept_names,
                l1_dim=l1_dim,
                l2_dim=l2_dim,
                kg_init_mode=kg_init_mode,
                kg_prior_map=kg_prior_map,
                kg_prior_strength=kg_prior_strength,
                action_mode=action_mode,
                action_prototypes=action_prototypes,
                prototype_and_weight=prototype_and_weight,
            )

        def _make_update_cell(logic_dim: int):
            act_dim = self.num_actions if self.use_action_feedback else None
            return ConceptUpdateCell(
                num_concepts=self.num_concepts,
                logic_dim=logic_dim,
                action_dim=act_dim,
                hidden_dim=256,
            )

        if self.shared_weights:
            self.base_reasoner = _make_reasoner()
            self.update_cell = _make_update_cell(self.base_reasoner.logic_dim)
        else:
            self.reasoners = nn.ModuleList([_make_reasoner() for _ in range(self.num_loops + 1)])
            self.update_cells = nn.ModuleList([
                _make_update_cell(self.reasoners[i].logic_dim) for i in range(self.num_loops)
            ])

    def get_beta(self) -> torch.Tensor:
        """Effective bounded refinement gate beta in [0, max_gate]."""
        return torch.sigmoid(self.raw_beta) * self.max_gate

    def kg_loss(self, concept_logits: torch.Tensor) -> torch.Tensor:
        if self.shared_weights:
            return self.base_reasoner.kg_loss(concept_logits)
        return self.reasoners[0].kg_loss(concept_logits)

    def rule_prior_loss(self) -> torch.Tensor:
        if self.shared_weights:
            return self.base_reasoner.rule_prior_loss()
        return torch.stack([r.rule_prior_loss() for r in self.reasoners]).mean()

    def rule_sparsity_loss(self) -> torch.Tensor:
        if self.shared_weights:
            return self.base_reasoner.rule_sparsity_loss()
        return torch.stack([r.rule_sparsity_loss() for r in self.reasoners]).mean()

    def forward(
        self,
        concept_logits: torch.Tensor,
        return_trace: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, List[torch.Tensor], List[Dict]]:
        """Forward pass executing K reasoning and refinement loops.

        Args:
            concept_logits: Initial concept logits z^0 [B, num_concepts]
            return_trace: Whether to record complete reasoning traces

        Returns:
            final_action_logits: y^K [B, num_actions]
            final_concept_logits: z^K [B, num_concepts]
            all_action_logits: list of action logits [y^0, y^1, ..., y^{K-1}]
            trace_history: list of dicts detailing each step
        """
        z_t = concept_logits
        all_action_logits = []
        trace_history = []
        beta = self.get_beta()

        for step in range(self.num_loops):
            reasoner = self.base_reasoner if self.shared_weights else self.reasoners[step]
            update_cell = self.update_cell if self.shared_weights else self.update_cells[step]

            y_t, logic_t, trace_t = reasoner(z_t, return_trace=return_trace)
            all_action_logits.append(y_t)

            delta_t = update_cell(
                z_t,
                logic_t,
                action_logits=y_t if self.use_action_feedback else None,
            )
            z_next = z_t + beta * delta_t

            if return_trace:
                trace_history.append({
                    "step": step,
                    "concept_logits": z_t.detach().cpu() if not self.training else z_t,
                    "concept_probs": torch.sigmoid(z_t).detach().cpu() if not self.training else torch.sigmoid(z_t),
                    "action_logits": y_t.detach().cpu() if not self.training else y_t,
                    "delta_z": delta_t.detach().cpu() if not self.training else delta_t,
                    "beta": beta.item(),
                    "rule_trace": trace_t,
                })

            z_t = z_next

        # Final reasoning step on refined concept logits z^K
        final_reasoner = self.base_reasoner if self.shared_weights else self.reasoners[self.num_loops]
        y_final, final_logic, final_rule_trace = final_reasoner(z_t, return_trace=return_trace)

        if return_trace:
            trace_history.append({
                "step": self.num_loops,
                "concept_logits": z_t.detach().cpu() if not self.training else z_t,
                "concept_probs": torch.sigmoid(z_t).detach().cpu() if not self.training else torch.sigmoid(z_t),
                "action_logits": y_final.detach().cpu() if not self.training else y_final,
                "delta_z": None,
                "beta": beta.item(),
                "rule_trace": final_rule_trace,
            })

        return y_final, z_t, all_action_logits, trace_history


class LCRSkateFormer(nn.Module):
    """Looping Concept Reasoning SkateFormer (LCR-SkateFormer).

    Integrates the SkateFormer perception backbone with the IterativeKGReasoner
    to perform multi-loop concept refinement and action reasoning.
    """

    def __init__(
        self,
        num_classes: int = 120,
        num_concepts: int = 72,
        spatial_concepts: int = 52,
        temporal_concepts: int = 20,
        num_people: int = 2,
        num_points: int = 24,
        num_frames: int = 64,
        backbone_weights: Optional[str] = None,
        concept_names: Optional[List[str]] = None,
        # Looping reasoning args
        num_loops: int = 2,
        shared_weights: bool = True,
        update_gate_init: float = 0.1,
        max_gate: float = 0.5,
        use_action_feedback: bool = False,
        # Reasoner args
        l1_dim: int = 128,
        l2_dim: int = 128,
        kg_init_mode: str = "kg_primed",
        kg_prior_map: Optional[Dict[str, List]] = None,
        kg_prior_strength: float = 1.0,
        action_mode: str = "linear",
        action_prototypes: Optional[Union[torch.Tensor, List]] = None,
        prototype_and_weight: float = 0.35,
        # Loss weights
        concept_loss_weight: float = 1.0,
        early_concept_loss_weight: float = 0.5,
        aux_action_loss_weight: float = 0.3,
        stability_loss_weight: float = 0.05,
        kg_loss_weight: float = 0.01,
        prior_retention_weight: float = 0.01,
        rule_sparsity_weight: float = 0.001,
        concept_negative_weight: float = 0.15,
        concept_positive_weight: float = 1.0,
        asymmetric_gamma_neg: float = 2.0,
        part_diversity_weight: float = 0.01,
        return_aux: bool = True,
        **kwargs,
    ):
        super().__init__()
        if int(spatial_concepts) + int(temporal_concepts) != int(num_concepts):
            raise ValueError("spatial_concepts + temporal_concepts must equal num_concepts")

        self.num_classes = int(num_classes)
        self.num_concepts = int(num_concepts)
        self.spatial_concepts = int(spatial_concepts)
        self.temporal_concepts = int(temporal_concepts)
        self.num_loops = int(num_loops)

        # Loss weights
        self.concept_loss_weight = float(concept_loss_weight)
        self.early_concept_loss_weight = float(early_concept_loss_weight)
        self.aux_action_loss_weight = float(aux_action_loss_weight)
        self.stability_loss_weight = float(stability_loss_weight)
        self.kg_loss_weight = float(kg_loss_weight)
        self.prior_retention_weight = float(prior_retention_weight)
        self.rule_sparsity_weight = float(rule_sparsity_weight)
        self.concept_negative_weight = float(concept_negative_weight)
        self.concept_positive_weight = float(concept_positive_weight)
        self.asymmetric_gamma_neg = float(asymmetric_gamma_neg)
        self.part_diversity_weight = float(part_diversity_weight)
        self.return_aux = bool(return_aux)
        self.accepts_concepts = True

        # Backbone
        self.backbone = SkateFormer_(
            num_classes=num_classes,
            num_people=num_people,
            num_points=num_points,
            num_frames=num_frames,
            **kwargs,
        )
        self._load_backbone_weights(backbone_weights)
        feature_dim = self.backbone.head.in_features

        # Concept Heads
        self.spatial_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.temporal_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.spatial_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, self.spatial_concepts),
        )
        self.temporal_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, self.temporal_concepts),
        )
        self.fusion = nn.Sequential(
            nn.Linear(2 * feature_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
        )

        if concept_names is None:
            concept_names = [f"concept_{i}" for i in range(self.num_concepts)]
        self.concept_names = list(concept_names)

        # Iterative KG Reasoner Engine
        self.reasoner = IterativeKGReasoner(
            num_concepts=self.num_concepts,
            num_actions=self.num_classes,
            concept_names=self.concept_names,
            num_loops=self.num_loops,
            shared_weights=shared_weights,
            update_gate_init=update_gate_init,
            max_gate=max_gate,
            use_action_feedback=use_action_feedback,
            l1_dim=l1_dim,
            l2_dim=l2_dim,
            kg_init_mode=kg_init_mode,
            kg_prior_map=kg_prior_map,
            kg_prior_strength=kg_prior_strength,
            action_mode=action_mode,
            action_prototypes=action_prototypes,
            prototype_and_weight=prototype_and_weight,
        )

    def _load_backbone_weights(self, checkpoint_path: Optional[str]):
        if not checkpoint_path:
            return
        path = Path(checkpoint_path)
        if not path.exists():
            print(f"[Warning] Backbone checkpoint not found: {path}. Initializing randomly.")
            return
        checkpoint = torch.load(str(path), map_location="cpu")
        if isinstance(checkpoint, dict):
            for key in ("state_dict", "model"):
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    checkpoint = checkpoint[key]
                    break
        if not isinstance(checkpoint, dict):
            raise TypeError("Unsupported SkateFormer checkpoint format")

        backbone_state = self.backbone.state_dict()
        mapped = OrderedDict()
        for key, value in checkpoint.items():
            clean_key = key.replace("module.", "", 1)
            if clean_key.startswith("backbone."):
                clean_key = clean_key[len("backbone."):]
            if clean_key.startswith("head."):
                continue
            if clean_key in backbone_state:
                mapped[clean_key] = value
        missing, unexpected = self.backbone.load_state_dict(mapped, strict=False)
        print(
            f"Loaded {len(mapped)} SkateFormer backbone tensors from {path}; "
            f"missing={len(missing)}, unexpected={len(unexpected)}"
        )

    def _pool_features(self, feature_map: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        spatial_tokens = feature_map.mean(dim=2)
        spatial_attention = torch.softmax(self.spatial_pool_score(spatial_tokens), dim=-1)
        spatial = (spatial_tokens * spatial_attention).sum(dim=-1)

        temporal_tokens = feature_map.mean(dim=3)
        temporal_attention = torch.softmax(self.temporal_pool_score(temporal_tokens), dim=-1)
        temporal = (temporal_tokens * temporal_attention).sum(dim=-1)
        return spatial, temporal

    @staticmethod
    def _part_diversity_loss(feature_map: torch.Tensor) -> torch.Tensor:
        batch, channels, frames, tokens = feature_map.shape
        if tokens % 6 != 0:
            return feature_map.new_zeros(())
        parts = feature_map.view(batch, channels, frames, 6, tokens // 6).mean(dim=(2, 4))
        parts = F.normalize(parts.transpose(1, 2), dim=-1)
        gram = torch.matmul(parts, parts.transpose(-1, -2))
        eye = torch.eye(gram.shape[-1], device=gram.device, dtype=gram.dtype)
        return ((gram - eye) ** 2).mean()

    def _concept_loss(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        target_mask: Optional[torch.Tensor] = None,
        target_confidence: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Asymmetric, confidence-aware loss for weak concept supervision."""
        targets = targets.float().clamp(0.0, 1.0)
        probabilities = torch.sigmoid(logits)
        positive_loss = F.softplus(-logits) * targets
        negative_loss = (
            F.softplus(logits)
            * (1.0 - probabilities).pow(self.asymmetric_gamma_neg)
            * (1.0 - targets)
        )
        weights = (
            targets * self.concept_positive_weight
            + (1.0 - targets) * self.concept_negative_weight
        )
        if target_mask is not None:
            weights = weights * target_mask.float().to(weights.device)
        if target_confidence is not None:
            weights = weights * target_confidence.float().to(weights.device)
        loss = positive_loss + negative_loss
        denominator = weights.sum().clamp_min(1.0)
        return (loss * weights).sum() / denominator

    def forward(
        self,
        input: torch.Tensor,
        index_t: Optional[torch.Tensor] = None,
        concept_targets: Optional[torch.Tensor] = None,
        concept_target_mask: Optional[torch.Tensor] = None,
        concept_target_confidence: Optional[torch.Tensor] = None,
        action_labels: Optional[torch.Tensor] = None,
        return_trace: bool = False,
    ) -> Union[torch.Tensor, Dict]:
        feature_map = self.backbone.forward_features_from_input(input, index_t=index_t)
        spatial_feat, temporal_feat = self._pool_features(feature_map)

        spatial_logits = self.spatial_head(spatial_feat)
        temporal_logits = self.temporal_head(temporal_feat)
        z0 = torch.cat([spatial_logits, temporal_logits], dim=1)

        # Looping KG reasoning
        y_final, z_final, all_actions, trace_history = self.reasoner(
            z0, return_trace=return_trace
        )

        if not self.return_aux and concept_targets is None and not self.training and not return_trace:
            return y_final

        # Calculate stability loss between successive concept states
        c0 = torch.sigmoid(z0)
        c_final = torch.sigmoid(z_final)
        stability_loss = (c_final - c0).abs().mean()

        output = {
            "action_logits": y_final,
            "concept_logits": z_final,
            "initial_concept_logits": z0,
            "spatial_logits": spatial_logits,
            "temporal_logits": temporal_logits,
            "all_action_logits": all_actions,
            "beta": self.reasoner.get_beta(),
            "fused_feature": self.fusion(torch.cat([spatial_feat, temporal_feat], dim=1)),
            "kg_loss": 0.5 * (self.reasoner.kg_loss(z0) + self.reasoner.kg_loss(z_final)),
            "prior_retention_loss": self.reasoner.rule_prior_loss(),
            "rule_sparsity_loss": self.reasoner.rule_sparsity_loss(),
            "part_diversity_loss": self._part_diversity_loss(feature_map),
            "stability_loss": stability_loss,
            "reasoning_trace": trace_history,
        }

        # Concept Supervision Losses
        if concept_targets is not None:
            targets = concept_targets.float().to(z0.device)
            if targets.shape[-1] != self.num_concepts:
                raise ValueError(
                    f"Expected concept targets with {self.num_concepts} labels, got {targets.shape[-1]}"
                )

            loss_c0 = self._concept_loss(
                z0,
                targets,
                target_mask=concept_target_mask,
                target_confidence=concept_target_confidence,
            )
            loss_cK = self._concept_loss(
                z_final,
                targets,
                target_mask=concept_target_mask,
                target_confidence=concept_target_confidence,
            )
            output["early_concept_loss"] = loss_c0
            output["final_concept_loss"] = loss_cK
            output["concept_loss"] = loss_cK

            # Auxiliary action loss if action_labels are provided in forward
            aux_act_loss = z0.new_zeros(())
            if action_labels is not None and len(all_actions) > 0:
                for act_t in all_actions:
                    aux_act_loss = aux_act_loss + F.cross_entropy(act_t, action_labels)
                aux_act_loss = aux_act_loss / len(all_actions)
            output["aux_action_loss"] = aux_act_loss

            output["aux_loss"] = (
                self.concept_loss_weight * loss_cK
                + self.early_concept_loss_weight * loss_c0
                + self.aux_action_loss_weight * aux_act_loss
                + self.stability_loss_weight * stability_loss
                + self.kg_loss_weight * output["kg_loss"]
                + self.prior_retention_weight * output["prior_retention_loss"]
                + self.rule_sparsity_weight * output["rule_sparsity_loss"]
                + self.part_diversity_weight * output["part_diversity_loss"]
            )

        return output


def LCR_SkateFormer(**kwargs):
    """Factory for LCR-SkateFormer."""
    return LCRSkateFormer(**kwargs)


def REASON_SkateFormer(**kwargs):
    """Backwards compatibility alias for REASON-SkateFormer."""
    return LCRSkateFormer(**kwargs)


def load_ecr_weights_into_lcr(lcr_model, ecr_checkpoint_path):
    print(f"Loading checkpoint from: {ecr_checkpoint_path}")
    checkpoint = torch.load(ecr_checkpoint_path, map_location="cpu")
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break

    raw_weights = OrderedDict([[k.split("module.")[-1], v] for k, v in checkpoint.items()])
    model_state = lcr_model.state_dict()
    mapped_weights = OrderedDict()

    for k, v in raw_weights.items():
        if k in model_state and model_state[k].shape == v.shape:
            mapped_weights[k] = v
        # Remap ECR reasoner -> LCR base_reasoner
        elif k.startswith("reasoner.") and not k.startswith("reasoner.base_reasoner."):
            new_k = k.replace("reasoner.", "reasoner.base_reasoner.", 1)
            if new_k in model_state and model_state[new_k].shape == v.shape:
                mapped_weights[new_k] = v
            else:
                mapped_weights[k] = v
        else:
            mapped_weights[k] = v

    missing, unexpected = lcr_model.load_state_dict(mapped_weights, strict=False)
    print(f"✅ Đã load thành công {len(mapped_weights)} tensors!")
    print(f"ℹ️ Tham số mới của Looping Reasoning (chưa có trong ECR): {len(missing)}")
    return lcr_model

# Đăng ký alias vào sys.modules cho package `model` và `model.lcr_skateformer`
# Giúp tương thích tuyệt đối với mọi code/script gọi `from model.lcr_skateformer import ...`
_model_mod = types.ModuleType("model")
_lcr_mod = types.ModuleType("model.lcr_skateformer")
_lcr_mod.KGUnionLayer = KGUnionLayer
_lcr_mod.KGPrimedReasoner = KGPrimedReasoner
_lcr_mod.ConceptUpdateCell = ConceptUpdateCell
_lcr_mod.IterativeKGReasoner = IterativeKGReasoner
_lcr_mod.LCRSkateFormer = LCRSkateFormer
_lcr_mod.LCR_SkateFormer = LCR_SkateFormer
_lcr_mod._logit = _logit
_lcr_mod.load_ecr_weights_into_lcr = load_ecr_weights_into_lcr

sys.modules["model"] = _model_mod
sys.modules["model.lcr_skateformer"] = _lcr_mod
sys.modules["model.reason_skateformer"] = _lcr_mod

print("✅ LCR-SkateFormer model classes (KGUnionLayer, IterativeKGReasoner, LCR_SkateFormer) loaded successfully!")



✅ LCR-SkateFormer model classes (KGUnionLayer, IterativeKGReasoner, LCR_SkateFormer) loaded successfully!


## ⚙️ Block 5: Multi-Objective Loss Functions


In [7]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = float(smoothing)

    def forward(self, x, target):
        confidence = 1.0 - self.smoothing
        logprobs = F.log_softmax(x, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        return (confidence * nll_loss + self.smoothing * smooth_loss).mean()

print("✅ Loss functions defined successfully!")



✅ Loss functions defined successfully!


## ⚡ Block 6: Engine Huấn luyện & Đánh giá (Auto-Resume & Checkpointing)


In [8]:
class Trainer:
    def __init__(self, model, train_loader, test_loader, optimizer, scheduler, loss_fn,
                 work_dir="./work_dir/lcr_exp", resume_path=None):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.loss_fn = loss_fn
        self.work_dir = Path(work_dir)
        self.resume_path = Path(resume_path) if resume_path else None
        self.work_dir.mkdir(parents=True, exist_ok=True)
        self.best_acc = 0.0
        self.best_epoch = 0
        self.start_epoch = 0
        self.global_step = 0

    def auto_resume(self):
        latest_pt = self.resume_path or (self.work_dir / "checkpoint_latest.pt")
        if latest_pt.exists():
            print(f"🔄 Tìm thấy checkpoint: {latest_pt}. Đang tiến hành Auto-Resume...")
            cp = torch.load(latest_pt, map_location=device)
            self.model.load_state_dict(cp['model'])
            self.optimizer.load_state_dict(cp['optimizer'])
            self.start_epoch = cp.get('epoch', 0)
            self.best_acc = cp.get('best_acc', 0.0)
            self.global_step = cp.get('global_step', 0)
            print(f"✅ Đã khôi phục thành công từ Epoch {self.start_epoch} (Best Acc: {self.best_acc:.2f}%)")

    def save_checkpoint(self, epoch, is_best=False):
        state = {
            'epoch': epoch + 1,
            'global_step': self.global_step,
            'best_acc': self.best_acc,
            'best_epoch': self.best_epoch,
            'model': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }
        torch.save(state, self.work_dir / "checkpoint_latest.pt")
        if is_best:
            torch.save(state, self.work_dir / "checkpoint_best.pt")
            print(f"⭐ [BEST] Đã lưu checkpoint tốt nhất tại epoch {epoch+1} (Acc = {self.best_acc:.2f}%)")

    def train_epoch(self, epoch):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(self.train_loader, desc=f"Train Epoch {epoch+1}")

        for batch in pbar:
            if len(batch) == 5:
                data, index_t, label, index, concept_targets = batch
            else:
                data, index_t, label, index = batch
                concept_targets = None

            data = data.float().to(device)
            index_t = index_t.float().to(device)
            label = label.long().to(device)
            if concept_targets is not None:
                concept_targets = concept_targets.float().to(device)

            if self.scheduler is not None:
                self.scheduler.step_update(self.global_step)
            self.global_step += 1

            output = self.model(data, index_t=index_t, concept_targets=concept_targets, action_labels=label)
            if isinstance(output, dict):
                act_logits = output["action_logits"]
                loss = self.loss_fn(act_logits, label) + output.get("aux_loss", 0.0)
            else:
                act_logits = output
                loss = self.loss_fn(act_logits, label)

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()
            _, pred = torch.max(act_logits.data, 1)
            correct += (pred == label).sum().item()
            total += label.size(0)

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{(correct/total)*100:.2f}%", "beta": f"{self.model.reasoner.get_beta().item():.3f}"})

        mean_loss = total_loss / len(self.train_loader)
        acc = (correct / total) * 100.0
        print(f"Epoch {epoch+1} Summary: Mean Loss = {mean_loss:.4f}, Train Acc = {acc:.2f}%")
        return mean_loss, acc

    def eval_epoch(self, epoch):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc=f"Eval Epoch {epoch+1}"):
                if len(batch) == 5:
                    data, index_t, label, index, _ = batch
                else:
                    data, index_t, label, index = batch

                data = data.float().to(device)
                index_t = index_t.float().to(device)
                label = label.long().to(device)

                output = self.model(data, index_t=index_t)
                act_logits = output["action_logits"] if isinstance(output, dict) else output
                loss = self.loss_fn(act_logits, label)

                total_loss += loss.item()
                _, pred = torch.max(act_logits.data, 1)
                correct += (pred == label).sum().item()
                total += label.size(0)

        acc = (correct / total) * 100.0
        is_best = acc > self.best_acc
        if is_best:
            self.best_acc = acc
            self.best_epoch = epoch + 1

        print(f"Eval Epoch {epoch+1}: Acc = {acc:.2f}% (Best: {self.best_acc:.2f}% @ Epoch {self.best_epoch})")
        self.save_checkpoint(epoch, is_best=is_best)
        return acc

    def fit(self, num_epochs=150, eval_interval=5):
        self.auto_resume()
        for epoch in range(self.start_epoch, num_epochs):
            self.train_epoch(epoch)
            if (epoch + 1) % eval_interval == 0 or (epoch + 1) == num_epochs:
                self.eval_epoch(epoch)



## 🧪 Block 7: Kiểm tra nhanh toàn diện (Smoke Test)


In [9]:
init_seed(1)
model = LCR_SkateFormer(
    num_classes=120,
    num_concepts=72,
    spatial_concepts=52,
    temporal_concepts=20,
    num_loops=2,
    shared_weights=True,
    update_gate_init=0.1,
    action_mode='linear'
).to(device)

x = torch.randn(2, 3, 64, 24, 2).to(device)
labels = torch.tensor([0, 1]).to(device)
targets = (torch.rand(2, 72) > 0.7).float().to(device)

out = model(x, concept_targets=targets, action_labels=labels, return_trace=True)
print("Action logits shape:", out["action_logits"].shape)
print("Concept logits shape:", out["concept_logits"].shape)
print("Total auxiliary loss:", out["aux_loss"].item())
print("Effective Beta Gate:", out["beta"].item())
print("Reasoning trace steps:", len(out["reasoning_trace"]))



C:\Users\ADMIN\miniconda3\envs\reason\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Action logits shape: torch.Size([2, 120])
Concept logits shape: torch.Size([2, 72])
Total auxiliary loss: 2.4157345294952393
Effective Beta Gate: 0.10000000149011612
Reasoning trace steps: 3


## 🔄 Block 8: Load Pretrained Checkpoint từ ECR-SkateFormer (Warm-up Phase 2)
> **Cơ chế Transfer Learning / Warm-up**:
> - `backbone`, `spatial_head`, `temporal_head`, `pooling`, `fusion`: Load nguyên vẹn 100%.
> - `reasoner.first`, `reasoner.second`, `reasoner.action` (từ ECR): Tự động remap vào `reasoner.base_reasoner.*` của LCR.
> - Các tham số mới của vòng lặp (`reasoner.raw_beta`, `reasoner.update_cell`): Được khởi tạo sẵn sàng cho quá trình tinh chỉnh $K$-loop.



In [10]:
# Checkpoint được load ở Block 9, sau khi `model` được khởi tạo.
# Đường dẫn sử dụng:
# D:/KLTN_Huy-Nhu/ECR-SkateFormer/work_dir/ntu/cs/REASON_SkateFormer_j/best_model.pt



## 🏋️ Block 9: Huấn luyện LCR-SkateFormer trên Dữ liệu Thật (NTU-RGB+D 60/120)
> Chạy block này để tiến hành train/fine-tune mô hình LCR-SkateFormer.



In [11]:
# 1. Khởi tạo Feeder với Dữ liệu Thật
DATA_PATH = "D:/KLTN_Huy-Nhu/CTR-GCN/data/ntu/NTU60_CS.npz"
CONCEPTS_CSV = CONCEPTS_CSV_PATH  # Sử dụng ontology đã nhúng sẵn

# Nếu có dữ liệu thực tế trên Drive/Local:
if os.path.exists(DATA_PATH):
    train_dataset = Feeder(
        data_path=DATA_PATH,
        concepts_csv=CONCEPTS_CSV,
        split="train",
        window_size=64,
        p_interval=[0.5, 1],
        aug_method="a123489",
        intra_p=0.5,
        inter_p=0.2,
        thres=64,
        uniform=True,
        partition=True,
    )

    test_dataset = Feeder(
        data_path=DATA_PATH,
        concepts_csv=CONCEPTS_CSV,
        split="test",
        window_size=64,
        p_interval=[0.95],
        thres=64,
        uniform=True,
        partition=True,
    )

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    # 2. Khởi tạo mô hình LCR-SkateFormer
    model = LCR_SkateFormer(
        num_classes=60,
        num_concepts=72,
        spatial_concepts=52,
        temporal_concepts=20,
        concept_names=train_dataset.concept_names,
        action_prototypes=train_dataset.concept_matrix[:60],
        num_loops=2,
        shared_weights=True,
        update_gate_init=0.1,
        action_mode="prototype",
        num_people=2,
        num_points=24,
        num_frames=64,
        num_heads=32,
        attn_drop=0.5,
        head_drop=0.0,
        rel=True,
        drop_path=0.2,
        type_1_size=(8, 8),
        type_2_size=(8, 12),
        type_3_size=(8, 8),
        type_4_size=(8, 12),
        mlp_ratio=4.0,
        index_t=True,
        backbone_weights=None,
    ).to(device)

    # Load pretrained ECR-SkateFormer weights before creating the optimizer.
    # Use forward slashes so this Windows path is not affected by Python escapes.
    ecr_ckpt = "D:/KLTN_Huy-Nhu/ECR-SkateFormer/work_dir/ntu/cs/REASON_SkateFormer_j/best_model.pt"
    print(f"ECR checkpoint path: {ecr_ckpt}")
    if not os.path.isfile(ecr_ckpt):
        raise FileNotFoundError(f"Không tìm thấy ECR checkpoint: {ecr_ckpt}")
    model = load_ecr_weights_into_lcr(model, ecr_ckpt)
    print("✅ Đã load ECR checkpoint thành công")

    # 3. Thiết lập Optimizer, Scheduler và Trainer
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.05)
    total_steps = 150 * len(train_loader)
    warmup_steps = 10 * len(train_loader)
    scheduler = CosineLRScheduler(optimizer, t_initial=total_steps, lr_min=1e-6, warmup_lr_init=1e-6, warmup_t=warmup_steps, t_in_epochs=False)
    loss_fn = LabelSmoothingCrossEntropy(smoothing=0.1)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        loss_fn=loss_fn,
        work_dir="./work_dir/ntu60_cs/LCR_SkateFormer_K2_proto",
        resume_path="D:/KLTN_Huy-Nhu/LCR-SkateFormer/work_dir/ntu60_cs/LCR_SkateFormer_K2_proto/checkpoint_latest.pt"
    )

    # 4. Bắt đầu huấn luyện
    trainer.fit(num_epochs=150, eval_interval=5)
else:
    print(f"ℹ️ Không tìm thấy file dataset tại {DATA_PATH}. Vui lòng cập nhật DATA_PATH phù hợp.")



ECR checkpoint path: D:/KLTN_Huy-Nhu/ECR-SkateFormer/work_dir/ntu/cs/REASON_SkateFormer_j/best_model.pt
Loading checkpoint from: D:/KLTN_Huy-Nhu/ECR-SkateFormer/work_dir/ntu/cs/REASON_SkateFormer_j/best_model.pt
✅ Đã load thành công 234 tensors!
ℹ️ Tham số mới của Looping Reasoning (chưa có trong ECR): 13
✅ Đã load ECR checkpoint thành công
🔄 Tìm thấy checkpoint: D:\KLTN_Huy-Nhu\LCR-SkateFormer\work_dir\ntu60_cs\LCR_SkateFormer_K2_proto\checkpoint_latest.pt. Đang tiến hành Auto-Resume...
✅ Đã khôi phục thành công từ Epoch 150 (Best Acc: 92.28%)


## 📊 Block 10: Phân tích Giải thích & Phản thực (Counterfactual Intervention)


In [12]:
model.eval()
with torch.no_grad():
    sample_x = torch.randn(1, 3, 64, 24, 2).to(device)
    out_orig = model(sample_x, return_trace=True)
    orig_prob = torch.softmax(out_orig["action_logits"], dim=-1)
    orig_top1 = torch.argmax(orig_prob).item()
    print(f"Original Top-1 Action: {orig_top1} with prob {orig_prob[0, orig_top1].item():.4f}")

    # Inspect reasoning trace across loops
    print("\n--- Looping Reasoning Trace ---")
    for step_info in out_orig["reasoning_trace"]:
        step = step_info["step"]
        beta = step_info["beta"]
        print(f"Step {step} | Beta: {beta:.4f}")



Original Top-1 Action: 35 with prob 0.0364

--- Looping Reasoning Trace ---
Step 0 | Beta: 0.3481
Step 1 | Beta: 0.3481
Step 2 | Beta: 0.3481


## 🔎 Block 11: Giải thích LCR trên mẫu test
Block này hiển thị action dự đoán, concept nổi bật qua từng vòng reasoning và kiểm tra can thiệp phản thực ở concept bottleneck.


In [13]:
# Chọn một mẫu từ test set để giải thích
model.eval()
batch = next(iter(test_loader))
if len(batch) == 5:
    data, index_t, label, sample_index, _ = batch
else:
    data, index_t, label, sample_index = batch

data = data[:1].float().to(device)
index_t = index_t[:1].float().to(device)
label_id = int(label[0].item())
concept_names = getattr(test_dataset, "concept_names", [f"concept_{i}" for i in range(72)])

with torch.no_grad():
    explanation = model(data, index_t=index_t, return_trace=True)

action_probs = torch.softmax(explanation["action_logits"], dim=-1)[0].detach().cpu()
pred_id = int(action_probs.argmax().item())
print(f"Ground-truth action id: {label_id}")
print(f"Predicted action id:    {pred_id} | confidence: {action_probs[pred_id].item():.4f}")

# 1. Action evidence sau từng vòng reasoning
print("\n--- Action evidence qua các vòng LCR ---")
for step_info in explanation["reasoning_trace"]:
    step = step_info["step"]
    step_probs = torch.softmax(step_info["action_logits"], dim=-1)[0].detach().cpu()
    step_pred = int(step_probs.argmax().item())
    beta = float(step_info["beta"])
    print(f"Loop {step}: action={step_pred:02d}, prob={step_probs[step_pred].item():.4f}, beta={beta:.4f}")

# 2. Các concept có xác suất cao nhất trước/sau reasoning
z0_prob = torch.sigmoid(explanation["initial_concept_logits"])[0].detach().cpu()
zk_prob = torch.sigmoid(explanation["concept_logits"])[0].detach().cpu()
delta_prob = zk_prob - z0_prob
top_idx = torch.topk(zk_prob, k=min(10, zk_prob.numel())).indices.tolist()
print("\n--- Top concepts sau reasoning ---")
print("{:<45s} {:>8s} {:>8s} {:>8s}".format("Concept", "z0", "zK", "delta"))
for i in top_idx:
    print(f"{concept_names[i]:45s} {z0_prob[i].item():8.4f} {zk_prob[i].item():8.4f} {delta_prob[i].item():+8.4f}")

# 3. Concept thay đổi mạnh nhất
change_idx = torch.topk(delta_prob.abs(), k=min(10, delta_prob.numel())).indices.tolist()
print("\n--- Concepts thay đổi mạnh nhất ---")
for i in change_idx:
    print(f"{concept_names[i]:45s} delta={delta_prob[i].item():+.4f}")

# 4. Can thiệp phản thực: đưa từng concept về trạng thái trung tính (logit = 0)
# So sánh xác suất của action gốc sau khi can thiệp tại bottleneck.
z0 = explanation["initial_concept_logits"].detach().clone()
baseline_pred_prob = action_probs[pred_id].item()
print("\n--- Counterfactual concept intervention ---")
print(f"Baseline action {pred_id}: {baseline_pred_prob:.4f}")
for i in top_idx[:5]:
    z_cf = z0.clone()
    z_cf[:, i] = 0.0
    with torch.no_grad():
        cf_logits, _, _, _ = model.reasoner(z_cf, return_trace=False)
    cf_probs = torch.softmax(cf_logits, dim=-1)[0].cpu()
    cf_pred = int(cf_probs.argmax().item())
    effect = cf_probs[pred_id].item() - baseline_pred_prob
    print(f"Ablate {concept_names[i]:35s} -> action={cf_pred:02d}, original-action delta={effect:+.4f}")


Ground-truth action id: 0
Predicted action id:    0 | confidence: 0.4997

--- Action evidence qua các vòng LCR ---
Loop 0: action=00, prob=0.4997, beta=0.3481
Loop 1: action=00, prob=0.4997, beta=0.3481
Loop 2: action=00, prob=0.4997, beta=0.3481

--- Top concepts sau reasoning ---
Concept                                             z0       zK    delta
arm_lower                                       1.0000   1.0000  +0.0000
leg_static_stand                                1.0000   1.0000  +0.0000
head_tilt_down                                  1.0000   1.0000  +0.0000
head_tilt_up                                    1.0000   1.0000  +0.0000
leg_alternating                                 1.0000   1.0000  +0.0000
foot_pivot                                      1.0000   1.0000  +0.0000
hand_grasp                                      1.0000   1.0000  +0.0000
hand_release                                    1.0000   1.0000  +0.0000
hand_wave                                       1.0000   1.0

In [14]:
z0 = explanation["initial_concept_logits"].detach().cpu()
zk = explanation["concept_logits"].detach().cpu()

print("z0 range:", z0.min().item(), z0.max().item())
print("zK range:", zk.min().item(), zk.max().item())
print("Max concept change:", (zk - z0).abs().max().item())

for info in explanation["reasoning_trace"]:
    dz = info["delta_z"]
    if dz is not None:
        print(
            f"Loop {info['step']}: "
            f"mean_delta={dz.abs().mean().item():.8f}, "
            f"max_delta={dz.abs().max().item():.8f}"
        )

z0 range: -44.03293991088867 102.8421630859375
zK range: -47.10199737548828 108.0463638305664
Max concept change: 5.61128044128418
Loop 0: mean_delta=3.87352920, max_delta=8.03653049
Loop 1: mean_delta=3.88405180, max_delta=8.08242702


In [15]:
z0 = explanation["initial_concept_logits"][0].detach().cpu()
zk = explanation["concept_logits"][0].detach().cpu()
delta = zk - z0

idx = torch.topk(delta.abs(), k=10).indices.tolist()

print("--- Concept logits thay đổi mạnh nhất ---")
for i in idx:
    print(
        f"{concept_names[i]:45s} "
        f"z0={z0[i].item():+.6f} "
        f"zK={zk[i].item():+.6f} "
        f"delta={delta[i].item():+.6f}"
    )

--- Concept logits thay đổi mạnh nhất ---
temporal_dynamics_rhythm_regular              z0=+24.653227 zK=+30.264507 delta=+5.611280
temporal_sequence_seq_hands_first             z0=+57.858570 zK=+63.413605 delta=+5.555035
foot_static                                   z0=+102.842163 zK=+108.046364 delta=+5.204201
arm_cross                                     z0=+54.313023 zK=+59.378620 delta=+5.065598
arm_hold                                      z0=+16.180872 zK=+20.901821 delta=+4.720949
temporal_direction_motion_upward              z0=+46.066498 zK=+50.735909 delta=+4.669411
temporal_dynamics_duration_brief              z0=+32.195728 zK=+36.864357 delta=+4.668629
foot_jump                                     z0=-20.890306 zK=-25.469099 delta=-4.578793
hand_circular                                 z0=+27.062040 zK=+31.354874 delta=+4.292833
foot_step                                     z0=+32.188622 zK=+36.453381 delta=+4.264759


In [16]:
proto = model.reasoner.base_reasoner.action_prototypes[0].detach().cpu()
z0 = explanation["initial_concept_logits"][0].detach().cpu()
zk = explanation["concept_logits"][0].detach().cpu()
delta = zk - z0

# Prototype=1 tăng là hỗ trợ.
# Prototype=0 giảm cũng là hỗ trợ.
alignment_delta = torch.where(proto > 0.5, delta, -delta)

support_idx = torch.topk(alignment_delta, k=10).indices
conflict_idx = torch.topk(-alignment_delta, k=10).indices

print("--- Concepts hỗ trợ action ---")
for i in support_idx:
    print(f"{concept_names[i]:45s} score={alignment_delta[i].item():+.4f}")

print("\n--- Concepts mâu thuẫn action ---")
for i in conflict_idx:
    print(f"{concept_names[i]:45s} score={alignment_delta[i].item():+.4f}")

--- Concepts hỗ trợ action ---
temporal_sequence_seq_hands_first             score=+5.5550
foot_static                                   score=+5.2042
temporal_direction_motion_upward              score=+4.6694
temporal_dynamics_duration_brief              score=+4.6686
foot_jump                                     score=+4.5788
hip_static_stand                              score=+3.8782
foot_kick                                     score=+3.8214
hip_sit                                       score=+3.6983
temporal_sequence_seq_alternating             score=+3.6708
arm_retract                                   score=+3.5435

--- Concepts mâu thuẫn action ---
temporal_dynamics_rhythm_regular              score=-5.6113
arm_cross                                     score=-5.0656
arm_hold                                      score=-4.7209
hand_circular                                 score=-4.2928
foot_step                                     score=-4.2648
temporal_dynamics_duration_sustain

## 📜 Block 12: Trích xuất luật cho một action
Hiển thị các literal có trọng số cao nhất trong nhánh AND/OR của KG reasoner.


In [25]:
# Chọn action cần trích xuất luật; mặc định dùng action đang được giải thích
action_id = 59
reasoner = model.reasoner.base_reasoner
if getattr(reasoner, "action_mode", "") != "prototype":
    raise ValueError("Block này cần action_mode='prototype'.")

action_rule = reasoner.action_rule
and_weights, or_weights = action_rule.weights()
and_weights = and_weights[:, action_id].detach().cpu()
or_weights = or_weights[:, action_id].detach().cpu()
n_concepts = len(concept_names)
prototype = reasoner.action_prototypes[action_id].detach().cpu()

action_name = f"action_{action_id}"
if "CONCEPTS_CSV" in globals() and os.path.isfile(CONCEPTS_CSV):
    with open(CONCEPTS_CSV, "r", newline="", encoding="utf-8-sig") as f:
        names = [row["action_class"] for row in csv.DictReader(f)]
    if action_id < len(names):
        action_name = names[action_id]

if "explanation" in globals():
    current_prob = torch.sigmoid(explanation["concept_logits"])[0].detach().cpu()
else:
    current_prob = None

def literal_name(index):
    if index < n_concepts:
        return concept_names[index]
    return "NOT_" + concept_names[index - n_concepts]

def literal_value(index):
    if current_prob is None:
        return float("nan")
    if index < n_concepts:
        return current_prob[index].item()
    return (1.0 - current_prob[index - n_concepts]).item()

def expected_literal(index):
    if index < n_concepts:
        return int(prototype[index].item())
    return int(1.0 - prototype[index - n_concepts].item())

def print_rule(title, weights):
    print(f"\n--- {title} ---")
    indices = torch.topk(weights, k=min(12, weights.numel())).indices.tolist()
    for index in indices:
        print(
            f"{literal_name(index):45s} "
            f"weight={weights[index].item():.6f} "
            f"value={literal_value(index):.6f} "
            f"expected={expected_literal(index)}"
        )

print(f"Action {action_id}: {action_name}")
print(f"Prototype positive concepts: {int(prototype.sum().item())}/{n_concepts}")
print("AND: các literal trọng số cao được xem như điều kiện kết hợp.")
print("OR: một literal trọng số cao có thể cung cấp evidence cho action.")
print_rule("AND rule literals", and_weights)
print_rule("OR rule literals", or_weights)
rule = explanation["reasoning_trace"][-1]["rule_trace"]

a = action_id
and_score = rule["and_score"][0, a].item()
or_score = rule["or_score"][0, a].item()
evidence = rule["evidence"][0, a].item()

print(f"AND contribution: {0.35 * and_score:.6f}")
print(f"OR contribution:  {0.65 * or_score:.6f}")
print(f"Final evidence:   {evidence:.6f}")

Action 59: walking_apart
Prototype positive concepts: 10/72
AND: các literal trọng số cao được xem như điều kiện kết hợp.
OR: một literal trọng số cao có thể cung cấp evidence cho action.

--- AND rule literals ---
leg_alternating                               weight=0.997154 value=1.000000 expected=1
temporal_direction_motion_backward            weight=0.995365 value=0.000000 expected=1
temporal_sequence_seq_alternating             weight=0.994121 value=0.000000 expected=1
interaction_two_person                        weight=0.991956 value=1.000000 expected=1
hip_sway                                      weight=0.985576 value=1.000000 expected=1
arm_swing                                     weight=0.953173 value=0.000000 expected=1
temporal_dynamics_duration_sustained          weight=0.950000 value=1.000000 expected=1
NOT_hip_squat                                 weight=0.941388 value=1.000000 expected=1
leg_step                                      weight=0.916811 value=1.000000 expe